## RSNA Pneumonia Detection Challenge with YOLO v8
We perform this task of detecting pneumonia with YOLO from ultralytics. Before that, we obtained the data in the form of jpeg for image and txt for labels, arranged them in the required YAML format.

### Dependencies
import the necessary libraries as usual.

In [ ]:
%%capture
import os
! pip install scikit-learn numpy matplotlib pandas ultralytics lightning torch 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torchvision import datasets, transforms
import pandas as pd
import numpy as np
from glob import glob
from PIL import Image
import logging
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

### YOLO
Train the model with the training data. We have split it into 7:3 training:validation

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
print("Training starts ")
model.train(
    data="/kaggle/input/datasets/poeticmage/rsna-pneumonia-detection-challenge-yaml/data.yaml",
    epochs=15,
    imgsz=640,
    verbose=False,
    stream=True
)
print("Training ends ")

### Testing
Test your model, save the bounded images

In [ ]:
print("Testing starts ")
results = model.predict(
    source="/kaggle/input/datasets/poeticmage/rsna-pneumonia-detection-challenge-jpeg-test/rsna_yolo/images/test",
    imgsz=768,
    conf=0.25,
    verbose=False,
    save=True
)
print("Testing ends ")

### Submission file
Arrange the submission file as asked.

In [ ]:
import os
from tqdm import tqdm

with open("submission.csv", "w") as file:
    file.write("patientId,PredictionString\n")

    for result in tqdm(results):
        patient_id = os.path.splitext(
            os.path.basename(result.path)
        )[0]

        out_str = patient_id + ","

        if result.boxes is not None and len(result.boxes) > 0:
            print("Pneumonia detected ",out_str)
            for box in result.boxes:
                conf = float(box.conf[0])

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                width=x2-x1
                height=y2-y1

                out_str += f" {round(conf, 2)} {int(x1)} {int(y1)} {int(width)} {int(height)}"

        file.write(out_str+"\n")

print("submission.csv created")